# Molecular simulation in Colab

Choose Runtime → Change runtime type → **L4 GPU** (or A100), then Runtime → Run all. The notebook selects GPU4PySCF automatically when an NVIDIA GPU is available and uses CPU on CPU runtimes. Set `COLABEXERCISE_DEVICE=gpu` to require GPU execution. CUDA packages install into an isolated Python environment, preserving the active notebook's packages. QE and browser downloads are opt-in. TPU is not supported. Download results or copy them to Drive before the runtime ends.

After the H₂ check, an optional section builds a **qubit Hamiltonian** (Jordan–Wigner or Bravyi–Kitaev), checks it against FCI in the right electron-number sector, and runs an exact ADAPT-VQE reference on it. To run your own configs on a Colab VM from a terminal instead, use `quantum-workflow-colab` (see `docs/colab.md`).

## Get the project
Use an existing checkout, upload the project to `/content/colabexercise`, or clone GitHub. `COLABEXERCISE_REF` can select a published branch; the default is `main`. Thread limits are set before importing numerical libraries.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import tempfile

for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):
    os.environ[key] = '1'
runtime_root = Path('/content') if Path('/content').is_dir() else Path(tempfile.gettempdir())
project = Path.cwd() if (Path.cwd() / 'quantum_workflow/workflow.py').is_file() else runtime_root / 'colabexercise'
repo_url = 'https://github.com/gutama/colabexercise.git'
branch = os.environ.get('COLABEXERCISE_REF', 'main')
if not project.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, repo_url, str(project)], check=True)
if not (project / 'pyproject.toml').is_file():
    raise RuntimeError('Project files are missing; upload the project or select its published branch.')
artifact_root = Path(os.environ.get('COLABEXERCISE_OUTPUT', str(runtime_root)))
artifact_root.mkdir(parents=True, exist_ok=True)
print('Python:', sys.version.split()[0], 'Project:', project)

In [ ]:
sys.path.insert(0, str(project))
from quantum_workflow.colab_runtime import prepare_environment, resolve_device

DEVICE = os.environ.get('COLABEXERCISE_DEVICE', 'auto')  # 'gpu' requires a GPU; 'cpu' selects CPU.
device = resolve_device(DEVICE)
runtime_python = prepare_environment(project, device, runtime_root / 'colabexercise-gpu-env')
print('SCF device:', device, 'Python:', runtime_python)

## Verify the NVIDIA GPU
CUDA 12 and 13 toolkits select separate, pinned GPU4PySCF/CuPy/cuTENSOR packages. The check below executes a CUDA operation and verifies cuTENSOR loading. Explicit GPU requests fail if hardware or dependencies are unavailable. SCF/DFT runs on GPU; full molecular-integral export and the Li-hBN NumPy demo use CPU.
CUDA 13 requires compute capability 8.0 or newer (L4/A100). T4 requires a CUDA 12 runtime.

In [ ]:
if device == 'gpu':
    subprocess.run([str(runtime_python), '-m', 'quantum_workflow.colab_runtime'], check=True)
else:
    print('CPU runtime selected. Choose a GPU runtime to enable GPU4PySCF.')

## PySCF: H₂ and integral export
GPU mode uses density fitting. For energy comparisons, use `density_fitting=true` on the CPU reference too. Change the example JSON for other systems; `summary.json` records the actual solver, GPU model, CUDA/package versions, and integral conventions.

In [ ]:
import json

config_data = json.loads((project / 'examples/h2-pyscf.json').read_text())
config_data['device'] = device
pyscf_output = Path(tempfile.mkdtemp(prefix='pyscf-', dir=artifact_root))
with tempfile.NamedTemporaryFile(mode='w', suffix='.json', dir=runtime_root, delete=False) as config_file:
    json.dump(config_data, config_file)
    config_path = Path(config_file.name)
try:
    subprocess.run([str(runtime_python), '-m', 'quantum_workflow.cli', str(config_path),
                    '--output', str(pyscf_output)], check=True)
finally:
    config_path.unlink()
summary = json.loads((pyscf_output / 'summary.json').read_text())
print(summary)
print('Output:', pyscf_output)

## Qubit Hamiltonian and ADAPT-VQE reference (CPU)
Runs a config with a `qubit` section, then an exact-statevector ADAPT-VQE on the exported `qubit_hamiltonian.json`. The run stops unless the reference bitstring reproduces the SCF energy and the sector ground state equals FCI/CASCI. HeH⁺ is the default because, unlike H₂, it exposes qubit-ordering bugs. Choose another example with `COLABEXERCISE_QUBIT_EXAMPLE` (`lih-bk.json`, `h2o-cas44.json`) or set it to `none` to skip. ADAPT here is a classical check of the pipeline, not a hardware run.

In [ ]:
QUBIT_EXAMPLE = os.environ.get('COLABEXERCISE_QUBIT_EXAMPLE', 'heh-cation-qubit.json')
if QUBIT_EXAMPLE != 'none':
    qubit_output = Path(tempfile.mkdtemp(prefix='qubit-', dir=artifact_root))
    subprocess.run([str(runtime_python), '-m', 'quantum_workflow.cli',
                    str(project / 'examples' / QUBIT_EXAMPLE), '--output', str(qubit_output)],
                   check=True, stdout=subprocess.DEVNULL)  # key numbers are printed below
    subprocess.run([str(runtime_python), '-m', 'quantum_workflow.adapt', str(qubit_output),
                    '--max-operators', '10'], check=True, stdout=subprocess.DEVNULL)
    qubit_summary = json.loads((qubit_output / 'summary.json').read_text())
    adapt = json.loads((qubit_output / 'adapt_vqe.json').read_text())
    q = qubit_summary['qubit_hamiltonian']
    print(f"{QUBIT_EXAMPLE}: {q['mapping']}, {q['n_qubits']} qubits, {q['n_terms']} Pauli terms, "
          f"{q['n_qwc_groups']} QWC groups")
    print(f"SCF {qubit_summary['energy_hartree']:.10f}  reference bits {q['reference_bits']} "
          f"-> {q['reference_energy_hartree']:.10f}")
    print(f"FCI/CASCI {qubit_summary['reference_energies_hartree'].get('fci')}  "
          f"sector ground state {q['sector_ground_energy_hartree']}")
    print(f"ADAPT-VQE {adapt['energy_hartree']:.10f} with {len(adapt['operators'])} operators; "
          f"error {adapt.get('error_hartree', float('nan')):.1e} Ha")
    if not adapt.get('chemical_accuracy'):
        raise RuntimeError('ADAPT-VQE did not reach chemical accuracy; inspect adapt_vqe.json')
    print('Output:', qubit_output)
else:
    print('Qubit section skipped.')

## Optional QE calculation (CPU)
Downloads micromamba and installs a separate conda-forge QE 7.6 CPU environment, plus an example H PBE pseudopotential. This takes several minutes and disk space. QE uses a periodic supercell; independently test vacuum and cutoffs. Review downloaded software and pseudopotential suitability and provenance before scientific use. This section does not export VQE-ready integrals.

In [ ]:
RUN_QE = os.environ.get('COLABEXERCISE_RUN_QE', '0') == '1'  # Set True to opt in.
if RUN_QE:
    from quantum_workflow.colab_remote import ensure_qe

    qe_bin = ensure_qe(runtime_root)  # micromamba + conda-forge QE 7.6, idempotent
    os.environ['PATH'] = str(qe_bin) + os.pathsep + os.environ['PATH']
    pseudo_dir = project / 'pseudopotentials'
    pseudo_dir.mkdir(exist_ok=True)
    pseudo = pseudo_dir / 'H.pbe-kjpaw_psl.1.0.0.UPF'
    subprocess.run(['curl', '-fL', 'https://pseudopotentials.quantum-espresso.org/upf_files/H.pbe-kjpaw_psl.1.0.0.UPF', '-o', str(pseudo)], check=True)
    qe_output = Path(tempfile.mkdtemp(prefix='qe-', dir=artifact_root))
    subprocess.run([str(runtime_python), '-m', 'quantum_workflow.cli', str(project / 'examples/h2-qe.json'),
                    '--output', str(qe_output)], check=True)
    print(json.loads((qe_output / 'summary.json').read_text()))
    print('Output:', qe_output)
else:
    print('QE skipped. Set RUN_QE = True to install QE and run the CPU example.')

## Download PySCF results (opt-in)
The archive includes integrals and metadata. Set `DOWNLOAD_RESULTS=True` to open a Colab browser download. For QE, archive `qe_output` similarly; its scratch directory can be large. See the README for tensor conventions and an FCI consumption example.

In [ ]:
DOWNLOAD_RESULTS = False  # True for a browser download in Google Colab.
if DOWNLOAD_RESULTS:
    import shutil
    from google.colab import files

    archive = shutil.make_archive(str(artifact_root / 'pyscf-results'), 'zip', pyscf_output)
    files.download(archive)
else:
    print('Results saved:', pyscf_output)